In [1]:
import numpy as np 
import pandas as pd 
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,SimpleRNN,Dense

In [2]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]

labels = [1]*15+[0]*15
labels = np.array(labels)

In [13]:
vocab_size = 2000
tok = Tokenizer(num_words=vocab_size,oov_token="")
tok.fit_on_texts(sentences)
seqs = tok.texts_to_sequences(sentences)
max_len = max(len(s) for s in seqs)
X = pad_sequences(seqs,maxlen=max_len,padding='post')
y = labels

In [14]:
max_len

5

In [15]:
X[0]

array([ 3, 26,  2,  7,  0])

In [16]:
embed_dim = 16
rnn_units = 8

In [17]:
inp = Input(shape=(max_len,),dtype="int32",name='input')
x = Embedding(input_dim=vocab_size,output_dim=embed_dim,mask_zero=True,name = 'embed')(inp)
rnn = SimpleRNN(units=rnn_units,return_sequences=False,return_state=False,name = 'simple_rnn')
x_last = rnn(x)
out = Dense(1,activation='sigmoid',name = 'out')(x_last)
model = Model(inputs = inp, outputs = out)
model.compile(optimizer='adam',loss = 'binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input (InputLayer)          [(None, 5)]               0         
                                                                 
 embed (Embedding)           (None, 5, 16)             32000     
                                                                 
 simple_rnn (SimpleRNN)      (None, 8)                 200       
                                                                 
 out (Dense)                 (None, 1)                 9         
                                                                 
Total params: 32,209
Trainable params: 32,209
Non-trainable params: 0
_________________________________________________________________


In [19]:
model.fit(X,y,epochs=25,batch_size=8,verbose=1)

Epoch 1/25
4/4 [==============================] - 2s 3ms/step - loss: 0.6993 - accuracy: 0.5000
Epoch 2/25
4/4 [==============================] - 0s 4ms/step - loss: 0.6861 - accuracy: 0.6000
Epoch 3/25
4/4 [==============================] - 0s 3ms/step - loss: 0.6744 - accuracy: 0.7333
Epoch 4/25
4/4 [==============================] - 0s 3ms/step - loss: 0.6635 - accuracy: 0.7667
Epoch 5/25
4/4 [==============================] - 0s 3ms/step - loss: 0.6523 - accuracy: 0.8000
Epoch 6/25
4/4 [==============================] - 0s 3ms/step - loss: 0.6408 - accuracy: 0.8333
Epoch 7/25
4/4 [==============================] - 0s 3ms/step - loss: 0.6275 - accuracy: 0.9000
Epoch 8/25
4/4 [==============================] - 0s 3ms/step - loss: 0.6151 - accuracy: 0.9333
Epoch 9/25
4/4 [==============================] - 0s 4ms/step - loss: 0.6007 - accuracy: 0.9333
Epoch 10/25
4/4 [==============================] - 0s 3ms/step - loss: 0.5857 - accuracy: 0.9333
Epoch 11/25
4/4 [======================

In [20]:
intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [22]:
from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(max_len,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)  # reuse trained embedding

# Create RNN with return_sequences=True
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# DO NOT CALL build() manually
seq_hidden = rnn_seq(seq_emb)  # builds automatically

# Copy trained RNN weights
try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Inspect
idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 [==============================] - 1s 581ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[-0.054 -0.001 -0.048  0.077  0.084  0.057  0.002 -0.1  ]
 [-0.023  0.055  0.191 -0.168  0.059  0.096 -0.096  0.073]
 [-0.246  0.155 -0.318  0.104 -0.214 -0.065  0.108 -0.146]
 [ 0.156 -0.071  0.324 -0.197  0.093  0.366  0.304  0.025]
 [ 0.156 -0.071  0.324 -0.197  0.093  0.366  0.304  0.025]]
